# Resample a diagnostic onto the common GWL grid

This notebook shows the operational use of `load_mapping` and `resample_to_gwl`:
turn a variable on **calendar time** into one indexed by **global warming level (GWL)**,
so models can be stacked on the shared 0–4 °C axis.

No external TIPMIP data are required — we use a bundled ramp-up mapping and a
synthetic annual diagnostic.

## Setup

Install the package from the repo root if you have not already:

```bash
pip install -e .
```

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr

from tipmip_gwl import load_mapping, list_models, resample_to_gwl
from tipmip_gwl.io import model_label

## Load a mapping product

Pick a model from the bundled ramp-up ensemble, or pass a path to your own `gwlmap_*.nc`.

In [ ]:
MODEL = "GFDL-ESM2M"  # change this, or set to None to use the first bundled model

if MODEL is None:
    models = list_models()
    if not models:
        raise RuntimeError("no bundled mappings found; install tipmip-gwl or pass a gwlmap path")
    MODEL = models[0]

mp = load_mapping(MODEL)
model = model_label(dict(mp.attrs))
ru_years = mp["year"].values

print(f"model: {model}")
print(f"ramp-up years: {int(ru_years.min())}–{int(ru_years.max())}")
print(f"max GWL reached: {float(mp['max_gwl_reached']):.2f} °C")

## Build a stand-in annual diagnostic

The year range is **intentionally offset** from the ramp-up (starts +5 years, ends
a few years early). `resample_to_gwl` aligns by coordinate *value*, not array
position, so this still works.

In [ ]:
diag_years = np.arange(ru_years.min() + 5, ru_years.max() - 2)
rng = np.random.default_rng(0)
values = np.cumsum(rng.standard_normal(diag_years.size)) + 50.0
diagnostic = xr.DataArray(
    values, dims="year", coords={"year": diag_years}, name="my_diagnostic"
)

print(
    f"diagnostic years: {int(diag_years.min())}–{int(diag_years.max())} "
    "(offset on purpose)"
)
diagnostic.plot(figsize=(8, 3), marker="o");

## Resample onto the common GWL grid

`resample_to_gwl` applies the inverse map `year_of_gwl(gwl)` and linearly
interpolates the diagnostic in calendar time at the fractional years returned.
Values are **never extrapolated** — GWLs the model never reached, or years
outside the diagnostic range, become NaN.

In [ ]:
on_gwl = resample_to_gwl(mp, diagnostic)
on_gwl

## Inspect a few GWL levels

In [ ]:
print(f"{'GWL':>5s} {'year_of_gwl':>12s} {'diagnostic(gwl)':>16s}")
for g in np.arange(0.0, 4.0001, 0.5):
    yr = float(mp["year_of_gwl"].sel(gwl=g, method="nearest"))
    val = float(on_gwl.sel(gwl=g, method="nearest"))
    yr_s = f"{yr:12.1f}" if np.isfinite(yr) else f"{'nan':>12s}"
    val_s = f"{val:16.3f}" if np.isfinite(val) else f"{'nan':>16s}"
    print(f"{g:5.1f} {yr_s} {val_s}")

In [ ]:
n_nan = int(np.isnan(on_gwl.values).sum())
print(
    f"{n_nan} of {on_gwl.size} GWL bins are NaN "
    "(unreached by the model, or outside the diagnostic's year range)."
)

## Next steps

- **`relabel_to_gwl`** — same diagnostic on each model's native GWL axis (no binning;
  good for single-model plots). See [using_mappings.md](../docs/using_mappings.md).
- **Ensemble stack** — loop over `list_models()`, `resample_to_gwl` each diagnostic,
  then `xr.concat(..., dim="model")`.
- **Your own NetCDF** — open with xarray; the annual coordinate must be calendar years
  (named `year` by default, or pass `year_dim=`).